# Connecting to ftp for reports

### Intro

use the code below to connect to INO observatory's ftp server and create a csv list of the list of files you need.

In [4]:
import subprocess
import re
import pandas as pd
from collections import defaultdict

# --- FTP configuration ---
HOST = '94.184.211.181'
PORT = 2121
USER = 'INOOBS'
PASS = 'INO123'

# --- Function to list directories/files on FTP ---
def list_ftp_dir(remote_dir):
    cmd = [
        "curl",
        "-k",
        "--ftp-ssl",
        "--ftp-pasv",
        f"ftp://{HOST}:{PORT}/{remote_dir.replace(' ', '%20')}/",
        "--user", f"{USER}:{PASS}",
        "-l"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    items = result.stdout.splitlines()
    return items

# --- Step 1: Navigate FTP directories interactively ---
current_dir = ""  # start at root
while True:
    items = list_ftp_dir(current_dir)
    print(f"\nContents of '{current_dir or '/'}':")
    for i, item in enumerate(items, start=1):
        print(f"{i}. {item}")

    print("b) Go BACK one directory")
    choice = input("\nEnter number of directory to enter, or 'd' if this is the directory you want: ").strip()
    
    if choice.lower() == 'd':
        break
    elif choice.lower() == 'b':
        if '/' in current_dir:
            current_dir = '/'.join(current_dir.split('/')[:-1])
        else:
            current_dir = ""
    else:
        try:
            idx = int(choice) - 1
            if 0 <= idx < len(items):
                current_dir = f"{current_dir}/{items[idx]}" if current_dir else items[idx]
            else:
                print("Invalid number.")
        except ValueError:
            print("Enter a number, 'd', or 'b'.")

print(f"\nSelected directory: {current_dir}")

# --- Step 2: Get list of all files in chosen directory ---
files = list_ftp_dir(current_dir)
print(f"\nFound {len(files)} files in selected directory.")

# --- Step 3: Define patterns for parsing ---
type_pattern = re.compile(r'(flat|dark|bias)', re.IGNORECASE)  # search anywhere
# Updated filter pattern for science frames (g, r, i, u, clear)
filter_pattern = re.compile(r'(?:^|_|-)(?P<filter>g|i|r|u|clear)(?:_|-|$)', re.IGNORECASE)
exptime_pattern = re.compile(r'exp(\d{2}\.\d{2}\.\d{2}\.\d+)_')
mode_pattern = re.compile(r'_(High|Low|Merge)_', re.IGNORECASE)
binning_pattern = re.compile(r'_(\d+x\d+)_', re.IGNORECASE)
date_pattern = re.compile(r'(\d{4}_\d{2}_\d{2})')

# --- Convert hh.mm.ss.fraction to seconds ---
def exptime_to_seconds(exptime_str):
    if exptime_str is None:
        return None
    parts = exptime_str.split('.')
    if len(parts) != 4:
        return None
    hh, mm, ss, frac_str = parts
    try:
        hh, mm, ss = int(hh), int(mm), int(ss)
        # Correctly handle microseconds: last part is fraction of second
        frac = float("0." + frac_str)
        return hh*3600 + mm*60 + ss + frac
    except ValueError:
        return None

# --- Step 4: Parse files ---
grouped = defaultdict(list)
for f in files:
    # --- Only High mode ---
    mode_match = mode_pattern.search(f)
    mode = mode_match.group(1).lower() if mode_match else None
    if mode != "high":
        continue

    # --- Type ---
    ftype_match = type_pattern.search(f)
    ftype = ftype_match.group(1).lower() if ftype_match else "science"

    # --- Filter ---
    filt_match = filter_pattern.search(f)
    filt = filt_match.group("filter").lower() if filt_match else None

    # --- Exposure time ---
    exptime_match = exptime_pattern.search(f)
    exptime_sec = exptime_to_seconds(exptime_match.group(1)) if exptime_match else None

    # --- Binning ---
    binning_match = binning_pattern.search(f)
    binning = binning_match.group(1) if binning_match else None

    # --- Date ---
    date_match = date_pattern.search(f)
    date = date_match.group(1) if date_match else None

    grouped[(ftype, filt, exptime_sec, binning)].append(f)

# --- Step 5: Build DataFrame rows ---
rows = []
for (ftype, filt, exptime, binning), file_list in grouped.items():
    n_files = len(file_list)
    row = {
        "Target": "",
        "Type": "",
        "Date": date,
        "Filter_Science": filt if ftype=="science" else None,
        "No# Science Frames": n_files if ftype=="science" else 0,
        "Binning_Science": binning if ftype=="science" else None,
        "Exp_science (sec)": exptime if ftype=="science" else None,
        "S/N": None,
        "Filter_Flat": filt if ftype=="flat" else None,
        "No# Flat Frames": n_files if ftype=="flat" else 0,
        "Binning_Flat": binning if ftype=="flat" else None,
        "Exp_Flat(sec)": exptime if ftype=="flat" else None,
        "Flat Quality": None,  # <-- left empty
        "No# Dark Frames": n_files if ftype=="dark" else 0,
        "Binning_Dark": binning if ftype=="dark" else None,
        "Exp_dark (sec)": exptime if ftype=="dark" else None,
        "No# Bias Frames": n_files if ftype=="bias" else 0,
        "Binning_Bias": binning if ftype=="bias" else None,
        "location": current_dir
    }
    rows.append(row)

df = pd.DataFrame(rows)

# --- ADD THESE 3 LINES ---
print("\nCOLUMNS PRESENT IN DF:")
print(df.columns.tolist())

print("\nFIRST FEW ROWS:")
print(df.head())
# --------------------------

df = df.sort_values(by=["Date", "Filter_Flat", "Exp_Flat(sec)"])

print("\nFinal summary table:")
print(df)

# --- Step 6: Ask user if they want to save ---
save_answer = input("\nDo you want to save this CSV? (y/n): ").strip().lower()
if save_answer in ['y', 'yes']:
    try:
        # If file exists, append without header
        df.to_csv("observation_summary.csv", mode='a', index=False, header=not pd.io.common.file_exists("observation_summary.csv"))
    except:
        df.to_csv("observation_summary.csv", index=False)
    print("Saved CSV as 'observation_summary.csv'.")
else:
    print("CSV not saved.")

# --- Step 7: Navigate up or quit ---
while True:
    cont = input("\nGo up one directory (u) or quit (q)? ").strip().lower()
    if cont == 'q':
        break
    elif cont == 'u':
        if '/' in current_dir:
            current_dir = '/'.join(current_dir.split('/')[:-1])
        else:
            current_dir = ""
        print(f"\nMoved up to: {current_dir or '/'}")

        # Show contents and select subdirectory
        while True:
            items = list_ftp_dir(current_dir)
            print(f"\nContents of '{current_dir or '/'}':")
            for i, item in enumerate(items, start=1):
                print(f"{i}. {item}")
            print("b) Go BACK one directory")

            choice = input("\nEnter number of directory to enter, or 'd' if this is the directory you want: ").strip()
            if choice.lower() == 'd':
                break
            elif choice.lower() == 'b':
                if '/' in current_dir:
                    current_dir = '/'.join(current_dir.split('/')[:-1])
                else:
                    current_dir = ""
            else:
                try:
                    idx = int(choice) - 1
                    if 0 <= idx < len(items):
                        current_dir = f"{current_dir}/{items[idx]}" if current_dir else items[idx]
                    else:
                        print("Invalid number.")
                except ValueError:
                    print("Enter a number, 'd', or 'b'.")

        # Re-parse and append CSV
        files = list_ftp_dir(current_dir)
        grouped = defaultdict(list)
        for f in files:
            # --- Only High mode ---
            mode_match = mode_pattern.search(f)
            mode = mode_match.group(1).lower() if mode_match else None
            if mode != "high":
                continue

            # --- Type ---
            ftype_match = type_pattern.search(f)
            ftype = ftype_match.group(1).lower() if ftype_match else "science"

            # --- Filter ---
            filt_match = filter_pattern.search(f)
            filt = filt_match.group("filter").lower() if filt_match else None

            # --- Exposure time ---
            exptime_match = exptime_pattern.search(f)
            exptime_sec = exptime_to_seconds(exptime_match.group(1)) if exptime_match else None

            # --- Binning ---
            binning_match = binning_pattern.search(f)
            binning = binning_match.group(1) if binning_match else None

            # --- Date ---
            date_match = date_pattern.search(f)
            date = date_match.group(1) if date_match else None

            grouped[(ftype, filt, exptime_sec, binning)].append(f)

        rows = []
        for (ftype, filt, exptime, binning), file_list in grouped.items():
            n_files = len(file_list)
            row = {
                "Target": "",
                "Type": "",
                "Date": date,
                "Filter_Science": filt if ftype=="science" else None,
                "No# Science Frames": n_files if ftype=="science" else 0,
                "Binning_Science": binning if ftype=="science" else None,
                "Exp_science (sec)": exptime if ftype=="science" else None,
                "S/N": None,
                "Filter_Flat": filt if ftype=="flat" else None,
                "No# Flat Frames": n_files if ftype=="flat" else 0,
                "Binning_Flat": binning if ftype=="flat" else None,
                "Exp_Flat(sec)": exptime if ftype=="flat" else None,
                "Flat Quality": None,
                "No# Dark Frames": n_files if ftype=="dark" else 0,
                "Binning_Dark": binning if ftype=="dark" else None,
                "Exp_dark (sec)": exptime if ftype=="dark" else None,
                "No# Bias Frames": n_files if ftype=="bias" else 0,
                "Binning_Bias": binning if ftype=="bias" else None,
                "location": current_dir
            }
            rows.append(row)

        df = pd.DataFrame(rows)
        try:
            df.to_csv("observation_summary.csv", mode='a', index=False, header=False)
        except:
            df.to_csv("observation_summary.csv", index=False)
        print("Appended CSV for new directory.")



Contents of '/':
1. Observations
2. Observations 2
b) Go BACK one directory



Enter number of directory to enter, or 'd' if this is the directory you want:  2



Contents of 'Observations 2':
1. 1404-07
b) Go BACK one directory



Enter number of directory to enter, or 'd' if this is the directory you want:  1



Contents of 'Observations 2/1404-07':
1. Altafi_09_29_2025
2. Altafi_10_13_2025
3. Altafi_10_19_2025
4. Altafi_Mahani_11_02_2025
5. Altafi_Shakeri_Karimi_10_15_2025
6. Mousavi_Sadr_11_22_2025
7. Rezaei-09-30-2025
8. Rezaei-10-01-2025
9. Rezaei_Altafi_10_07_2025
10. Rezaei_Amiri_2025_11_18
11. rezaei_amiri_joroghi_2025_11_11
12. Rezaei_Hossein_Atanaz_Kosar_2025_11_04
13. rezaei_saba_farideH_2025_10_21
14. rezaei_saba_farideH_2025_10_22
b) Go BACK one directory


...


Enter number of directory to enter, or 'd' if this is the directory you want:  2



Contents of 'Observations 2/1404-07/Mousavi_Sadr_11_22_2025/Lockman Hole':
1. r_2025_11_23_1x1_exp00.01.40.000_000001_High_1.fit
2. r_2025_11_23_1x1_exp00.01.40.000_000001_Low_1.fit
3. r_2025_11_23_1x1_exp00.01.40.000_000001_Merge_1.fit
4. r_2025_11_23_1x1_exp00.01.40.000_000002_High_1.fit
5. r_2025_11_23_1x1_exp00.01.40.000_000002_Low_1.fit
6. r_2025_11_23_1x1_exp00.01.40.000_000002_Merge_1.fit
7. r_2025_11_23_1x1_exp00.01.40.000_000003_High_1.fit
8. r_2025_11_23_1x1_exp00.01.40.000_000003_Low_1.fit
9. r_2025_11_23_1x1_exp00.01.40.000_000003_Merge_1.fit
10. r_2025_11_23_1x1_exp00.01.40.000_000004_High_1.fit
11. r_2025_11_23_1x1_exp00.01.40.000_000004_Low_1.fit
12. r_2025_11_23_1x1_exp00.01.40.000_000004_Merge_1.fit
13. r_2025_11_23_1x1_exp00.03.20.000_000001_High_1.fit
14. r_2025_11_23_1x1_exp00.03.20.000_000001_High_2.fit
15. r_2025_11_23_1x1_exp00.03.20.000_000001_High_3.fit
16. r_2025_11_23_1x1_exp00.03.20.000_000001_Low_1.fit
17. r_2025_11_23_1x1_exp00.03.20.000_000001_Low_2.fit



Enter number of directory to enter, or 'd' if this is the directory you want:  d


Appended CSV for new directory.



Go up one directory (u) or quit (q)?  q
